# First look at the data

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "analysis/__init__.py").is_file())
ANALYSIS = PROJECT / "analysis"
OUTPUT = ANALYSIS / "outputs" / "first_look"
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(ANALYSIS / ".cache" / "matplotlib"))
sys.path.insert(0, str(PROJECT))
from analysis.data.first_look import (source_inventory, extract_pilot, summarize_pilot,
                        meeting_notes, CANDIDATE_FEATURES)
from analysis.evaluation.first_look_plots import make_plots
print("Project:", PROJECT)
print("Python:", sys.version.split()[0])
print("Outputs:", OUTPUT)

## 1. Reading the data

- The first 1,000 rows from each LLP file.
- The first 6 rows from each unpaired file.

This keeps the first check small. It uses the start of each file, so it is not a random sample.

In [ ]:
inventory = source_inventory(PROJECT, OUTPUT)
file_summary = pd.DataFrame(inventory["group_summary"])
display(file_summary[["sample", "files", "total_entries", "schema_variants",
                      "pilot_rows_per_file", "planned_pilot_rows"]])

## 2. Check the data

Codex says: The saved run read **3,746 rows containing 25,524 jets**. All these count and list checks passed. A `0` in a `_failures` column means no problem was found by that check.

I still need to check what one row represents and which cells belong to each jet.

In [ ]:
jets, rows, cells, examples = extract_pilot(PROJECT, inventory, OUTPUT)
summary = summarize_pilot(jets, rows, cells, examples, inventory, OUTPUT)
counts = pd.DataFrame(summary["groups"])
display(counts[["sample", "files_read", "rows_read", "jets", "empty_jet_rows",
                "rows_with_multiple_jets", "repeated_event_keys"]])
check_columns = [c for c in counts.columns if c.endswith("_failures")]
display(counts[["sample"] + check_columns])
display(Markdown("**Row interpretation:** " + summary["row_interpretation"]))

## 3. Look at two examples

I take the first row with cells from each group:

| Files | Jets in the row | Cells in the row |
|---|---:|---:|
| LLP | 13 | 922 |
| Unpaired | 1 | 4 |

These are just examples, not necessarily typical rows.

In the jet tables, `jet_pt` is momentum across the beam direction, `jet_eta` describes direction, and `jet_time_ATLAS` is the stored jet time. `jet_time_spread` measures how much the times vary; I need to check its exact definition.

I also need to confirm the units and timing corrections with Emil.


In [ ]:
example_table = pd.DataFrame(examples)
display(example_table[["sample", "source_file", "row_index", "runNumber",
                       "lumiBlock", "eventNumber", "jet_count", "cluster_count",
                       "cell_count", "finite_time_z_pairs"]])
for example in examples:
    selected = ((jets["sample"] == example["sample"]) &
                (jets["source_file"] == example["source_file"]) &
                (jets["row_index"] == example["row_index"]))
    display(Markdown("**" + example["sample"] + ": jets stored in the selected row**"))
    display(jets.loc[selected, ["jet_index", "jet_pt", "jet_eta", "jet_time_ATLAS",
                                 "jet_time_spread", "jet_n_clusters"]].head(15))

## 4. Plot the jets

In [ ]:
figures = make_plots(jets, cells, OUTPUT)
display(Image(filename=figures["01_jet_kinematics"]))
display(Image(filename=figures["02_jet_timing"]))

## 5. Plot the cells

Now each point is a **cell**. Across is `cell_z`, its position along the beam. Up is `cell_time`, its stored time.

The left plot has all 922 cells from a row with 13 jets. I have not separated the cells by jet.

The right plot has only 4 cells. This is too little to show a typical BIB pattern. The units still need checking.


In [ ]:
display(Image(filename=figures["03_cell_timing"]))

In [ ]:
display(pd.DataFrame(summary["jet_value_quality"])[
    ["sample", "branch", "values", "nonfinite", "minus_999999", "negative_finite", "zero",
     "min", "median", "max"]])
print("Candidate reconstructed inputs, after definitions are confirmed:")
print(", ".join(CANDIDATE_FEATURES))
print("Equal recorded run/lumi/event keys across sources:",
      len(summary["same_recorded_event_keys_across_sources"]))
display(counts[["sample", "truth_match_flag_available_jets", "truth_match_flag_equals_one"]])
print("jet_is_llp_matched is a diagnostic here; no truth selection has been applied.")
display(Markdown("**Observed finite sentinel-like values:** `jet_BIB_score` and the fit fields contain `-999999`. These need documented missing-value handling; they have not been cut or used as truth."))
print("The first unpaired cell example contains only four cells; it is not a representative BIB pattern.")